# Week 2: Post-Class Exercise - Titanic Survival Classification

**Dataset:** Titanic Passenger Data  
**Goal:** Predict passenger survival based on demographics and ticket information  
**Scaffolding Level:** 50% provided (homework practice)  
**Time Estimate:** 60-90 minutes

---

## Learning Objectives

By completing this exercise, you will:
1. Build a complete classification pipeline independently
2. Handle categorical features (encoding Sex, Embarked)
3. Apply all metrics learned in class (accuracy, precision, recall, F1, AUC)
4. Make decisions about which metrics to optimize
5. Analyze model performance across different passenger groups

---

## The Challenge

You are a data scientist consulting for a modern cruise line company. They want to understand historical survival patterns from the Titanic disaster to improve their safety protocols and resource allocation.

**Business context:**
- **False Negative (FN):** Passenger died but model predicted survival → **Inadequate safety measures!**
- **False Positive (FP):** Passenger survived but model predicted death → Overspending on safety equipment

**Question to keep in mind:** Which error is worse from a safety perspective?

---

## Dataset Information

**Features:**
- `Pclass`: Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)
- `Sex`: Male or female
- `Age`: Age in years (may have missing values)
- `SibSp`: Number of siblings/spouses aboard
- `Parch`: Number of parents/children aboard
- `Fare`: Passenger fare
- `Embarked`: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)

**Target:**
- `Survived`: 1 = survived, 0 = died

**Sample size:** 891 passengers

**Historical context:** On April 15, 1912, the "unsinkable" RMS Titanic sank after hitting an iceberg. Of the 2,224 passengers and crew, only about 32% survived. Survival was not random - class, gender, and age played significant roles.

---

Let's begin!

---

## Part 1: Setup and Data Loading

### Task 1.1: Imports

**TODO:** Import all necessary libraries.

**Hint:** Review your Day 2 guided practice or sklearn cheat sheet!

In [ ]:
# TODO: Import all necessary libraries
# Hint: You need pandas, numpy, matplotlib, sklearn modules

# Data handling
import ____ as np
import ____ as pd

# sklearn - preprocessing and model selection
from sklearn.model_selection import ____
from sklearn.preprocessing import ____

# sklearn - model
from sklearn.linear_model import ____

# sklearn - metrics
from sklearn.metrics import (
    ____,  # accuracy
    ____,  # confusion matrix
    ____,  # classification report
    ____,  # precision
    ____,  # recall
    ____,  # f1-score
    ____,  # ROC curve
    ____   # AUC score
)

# Visualization
import ____.pyplot as plt
import ____ as sns

# Jupyter magic command
%matplotlib inline

print("✅ All imports successful!")

### Task 1.2: Load Data

**TODO:** Load the Titanic dataset.

**Note:** We'll use seaborn's built-in Titanic dataset for convenience!

In [ ]:
# Load Titanic data from seaborn
# This is a clean version of the famous Kaggle dataset

df = sns.load_dataset('titanic')

print("✅ Data loaded!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

### Task 1.3: Data Exploration

**TODO:** Explore the dataset to understand its characteristics.

In [ ]:
# TODO: Display basic information about the dataset
# Hint: Use df.info() and df.describe()

print("Dataset Information:")
df.____()  # Show data types and non-null counts

print("\nBasic Statistics:")
print(df.____())  # Summary statistics for numeric columns

In [ ]:
# TODO: Check for missing values
# Hint: df.isnull().sum()

print("Missing Values:")
print(df.____.____())

In [ ]:
# Check target distribution (provided)

print("Survival distribution:")
print(df['survived'].value_counts().sort_index())
print(f"\nSurvival rate: {df['survived'].mean()*100:.1f}%")
print(f"  Died (0):     {(df['survived']==0).sum()} ({(df['survived']==0).sum()/len(df)*100:.1f}%)")
print(f"  Survived (1): {(df['survived']==1).sum()} ({(df['survived']==1).sum()/len(df)*100:.1f}%)")

**Question 1.1:** Is this dataset balanced or imbalanced? What does this mean for our evaluation?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

### Task 1.4: Feature Engineering

**TODO:** Prepare features for modeling by encoding categorical variables and handling missing values.

**Strategy:**
1. Select relevant features
2. Handle missing values
3. Encode categorical variables (Sex, Embarked)

In [ ]:
# TODO: Select features and create a clean dataset
# Hint: Keep only: pclass, sex, age, sibsp, parch, fare, embarked, survived
# Expected: A new DataFrame with these columns

# Select relevant columns
features_to_keep = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df_clean = df[____].copy()  # Fill in the features to keep

print(f"✅ Selected features!")
print(f"Shape: {df_clean.shape}")
print(f"\nColumns: {list(df_clean.columns)}")

In [ ]:
# TODO: Handle missing values
# Hint: For Age - fill with median, For Embarked - fill with mode (most common)
# Expected: No missing values in age or embarked

# Fill missing Age with median
df_clean['age'].fillna(df_clean['age'].____(), inplace=True)  # Fill in the method

# Fill missing Embarked with mode (most common value)
df_clean['embarked'].fillna(df_clean['embarked'].____()[0], inplace=True)  # Fill in the method

# Drop any remaining rows with missing values
df_clean = df_clean.dropna()

print(f"✅ Missing values handled!")
print(f"Remaining rows: {len(df_clean)}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

In [ ]:
# TODO: Encode categorical variables
# Hint: Sex (male=1, female=0), Embarked (create dummy variables)
# Expected: All numeric features ready for modeling

# Encode Sex: male=1, female=0
df_clean['sex'] = (df_clean['sex'] == '____').astype(int)  # Fill in the gender

# One-hot encode Embarked (creates dummy columns)
df_clean = pd.get_dummies(df_clean, columns=['____'], drop_first=True)  # Fill in column name

print("✅ Categorical variables encoded!")
print(f"\nFinal columns:")
print(list(df_clean.columns))
print(f"\nFirst few rows:")
df_clean.head()

### Task 1.5: Separate Features and Target

**TODO:** Split the data into X (features) and y (target).

In [ ]:
# TODO: Separate features (X) and target (y)
# Hint: X should have all columns except 'survived', y should have only 'survived'
# Expected: X has 8 columns (after encoding), y has 1 dimension

X = df_clean.____('survived', axis=1).values  # Drop target column
y = df_clean['____'].values                   # Target column only

print(f"✅ Features and target separated!")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

---

## Part 2: Data Preparation

### Task 2.1: Train/Test Split

**TODO:** Split the data into training and test sets.

**Remember:** Split BEFORE preprocessing!

In [ ]:
# TODO: Split data into 80% train, 20% test
# Hint: Use train_test_split with test_size, random_state, and stratify
# Expected: Training set ~570 samples, test set ~140 samples

X_train, X_test, y_train, y_test = ____(
    X, y,
    test_size=____,      # 20% for testing
    random_state=____,   # For reproducibility (use 42)
    stratify=____        # Maintain class balance
)

print("✅ Data split complete!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(f"  Died:     {(y_train==0).sum()}")
print(f"  Survived: {(y_train==1).sum()}")

### Task 2.2: Feature Scaling

**TODO:** Scale the features using StandardScaler.

**Critical:** Fit on training data only!

In [ ]:
# TODO: Create StandardScaler and scale features
# Hint: fit_transform on training, transform on test
# Expected: Mean ≈ 0, Std ≈ 1 for training data

# Create scaler
scaler = ____()  # Fill in the scaler class

# Fit on training data and transform
X_train_scaled = scaler.____(____)  # Fit AND transform

# Transform test data (using training statistics!)
X_test_scaled = scaler.____(____)   # Transform only

print("✅ Features scaled!")
print(f"\nTraining data (first feature):")
print(f"  Mean: {X_train_scaled[:, 0].mean():.4f}")
print(f"  Std:  {X_train_scaled[:, 0].std():.4f}")

---

## Part 3: Model Training

### Task 3.1: Create and Train Model

**TODO:** Train a logistic regression classifier.

In [ ]:
# TODO: Create and train logistic regression model
# Hint: LogisticRegression(max_iter=10000, random_state=42)
# Expected: Model trained successfully

# Create model
model = ____(
    max_iter=____,
    random_state=____
)

# Train model
model.____(____, ____)  # Fit the model

print("✅ Model trained!")

### Task 3.2: Make Predictions

**TODO:** Generate both class and probability predictions.

In [ ]:
# TODO: Make predictions on test set
# Hint: Use both predict() and predict_proba()
# Expected: y_pred has class labels (0/1), y_pred_proba has probabilities

# Class predictions
y_pred = model.____(____)  # Predict classes

# Probability predictions
y_pred_proba = model.____(____)  # Predict probabilities

print("✅ Predictions made!")
print(f"\nFirst 5 predictions:")
print(f"  Classes: {y_pred[:5]}")
print(f"  Probabilities (survived): {y_pred_proba[:5, 1]}")

---

## Part 4: Model Evaluation

### Task 4.1: Calculate Basic Metrics

**TODO:** Calculate accuracy, precision, recall, and F1-score.

In [ ]:
# TODO: Calculate all basic metrics
# Hint: Use accuracy_score, precision_score, recall_score, f1_score
# Expected: Four numbers between 0 and 1

accuracy = ____(____, ____)   # Calculate accuracy
precision = ____(____, ____)  # Calculate precision
recall = ____(____, ____)     # Calculate recall
f1 = ____(____, ____)         # Calculate F1-score

print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)
print(f"\nAccuracy:  {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"Precision: {precision:.3f} ({precision*100:.1f}%)")
print(f"Recall:    {recall:.3f} ({recall*100:.1f}%)")
print(f"F1-Score:  {f1:.3f} ({f1*100:.1f}%)")
print("=" * 60)

**Question 4.1:** For Titanic survival prediction, which metric is most important? Why?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

___________________________________________________________________

### Task 4.2: Confusion Matrix

**TODO:** Generate and visualize the confusion matrix.

In [ ]:
# TODO: Generate confusion matrix
# Hint: Use confusion_matrix(y_test, y_pred)
# Expected: 2x2 array

cm = ____(____, ____)  # Generate confusion matrix

print("Confusion Matrix:")
print(cm)
print("\nBreakdown:")
tn, fp, fn, tp = cm.ravel()
print(f"  True Negatives (TN):  {tn}")
print(f"  False Positives (FP): {fp}")
print(f"  False Negatives (FN): {fn}")
print(f"  True Positives (TP):  {tp}")

In [ ]:
# TODO: Visualize confusion matrix as heatmap
# Hint: Use seaborn heatmap with annot=True, fmt='d'
# Expected: A colored 2x2 grid with counts

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=____,          # Show numbers (True/False)
    fmt='____',          # Integer format ('d')
    cmap='____',         # Color scheme (try 'Blues', 'viridis', 'RdYlGn')
    xticklabels=['Died', 'Survived'],
    yticklabels=['Died', 'Survived'],
    cbar_kws={'label': 'Count'}
)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix: Titanic Survival Prediction')
plt.tight_layout()
plt.show()

**Question 4.2:** Interpret the confusion matrix:

a) How many passengers who survived were correctly identified?

*Your answer:* _______________

b) How many passengers who died were incorrectly predicted to survive (false negatives)?

*Your answer:* _______________

c) In the cruise line safety context, which is more concerning - predicting survival for someone who died (FN) or predicting death for someone who survived (FP)? Why?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

### Task 4.3: Classification Report

**TODO:** Generate a comprehensive classification report.

In [ ]:
# TODO: Generate classification report
# Hint: Use classification_report() with target_names parameter
# Expected: Detailed table with precision, recall, F1 for each class

print("Classification Report:")
print(____(____, ____, target_names=['Died', 'Survived']))  # Fill in function and parameters

### Task 4.4: ROC Curve and AUC

**TODO:** Calculate and plot the ROC curve.

In [ ]:
# TODO: Calculate ROC curve and AUC
# Hint: Use roc_curve() and roc_auc_score()
# Note: Use probabilities of positive class (survived = 1)
# Expected: fpr, tpr arrays and AUC score between 0.5 and 1.0

fpr, tpr, thresholds = ____(____, ____[:, 1])  # ROC curve
auc = ____(____, ____[:, 1])                   # AUC score

print(f"AUC Score: {auc:.3f}")

In [ ]:
# TODO: Plot ROC curve
# Hint: Plot TPR vs FPR, include diagonal reference line
# Expected: Curve above diagonal (better than random)

plt.figure(figsize=(8, 6))

# Plot ROC curve
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {auc:.3f})')

# Plot diagonal (random guessing)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
         label='Random Guessing (AUC = 0.5)')

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve: Titanic Survival Prediction')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Question 4.3:** Interpret the AUC score:

a) What does your AUC score mean?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

b) Is this considered excellent, good, fair, or poor? (Refer to Week 2 materials)

*Your answer:* _______________

---

## Part 5: Analysis and Insights

### Task 5.1: Compare Metrics Summary

**TODO:** Create a summary table comparing all metrics.

In [ ]:
# TODO: Create a summary DataFrame of all metrics
# Hint: Use pd.DataFrame() with a dictionary
# Expected: Clean table with metric names and values

metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
    'Score': [____, ____, ____, ____, ____],  # Fill in the calculated values
    'Interpretation': [
        'Overall correctness',
        'Accuracy of survival predictions',
        'Percentage of survivors identified',
        'Balance of precision and recall',
        'Overall discrimination ability'
    ]
})

print("\n" + "="*70)
print("METRICS SUMMARY")
print("="*70)
print(metrics_summary.to_string(index=False))
print("="*70)

### Task 5.2: Survival Analysis by Passenger Group

**TODO:** Analyze which passenger groups are harder to predict.

In [ ]:
# Reconstruct test data with predictions for analysis (PROVIDED)

# Get original test indices
test_indices = df_clean.index[-len(y_test):]
test_df = df_clean.loc[test_indices].copy()
test_df['predicted'] = y_pred
test_df['correct'] = (test_df['survived'] == test_df['predicted'])

# Decode sex back to readable form for analysis
test_df['sex_label'] = test_df['sex'].map({1: 'Male', 0: 'Female'})

print("✅ Test data prepared for analysis")

In [ ]:
# Analyze prediction accuracy by passenger class (PROVIDED)

print("Prediction Accuracy by Passenger Class:")
print(test_df.groupby('pclass')['correct'].agg(['sum', 'count', 'mean']))
print("\nInterpretation: Higher 'mean' = better predictions for that class")

In [ ]:
# Analyze prediction accuracy by gender (PROVIDED)

print("Prediction Accuracy by Gender:")
print(test_df.groupby('sex_label')['correct'].agg(['sum', 'count', 'mean']))
print("\nInterpretation: Are males or females easier to predict?")

**Question 5.1:** Based on the analysis above:

a) Which passenger class is easiest to predict?

*Your answer:* _______________

b) Which gender is easier to predict?

*Your answer:* _______________

c) Why do you think this is the case? (Think about historical context - "women and children first")

*Your answer:*

___________________________________________________________________

___________________________________________________________________

---

## Part 6: Critical Thinking Questions

### Question 6.1: Business Context

**Scenario:** You're presenting this model to modern cruise line executives.

a) Which metric would you emphasize most for improving safety protocols? Why?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

___________________________________________________________________

b) How would you explain a false negative to a non-technical executive?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

___________________________________________________________________

c) Based on your model's performance, what safety recommendations would you make?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

___________________________________________________________________

### Question 6.2: Threshold Tuning

a) If you lowered the threshold from 0.5 to 0.3, what would happen to recall? What about precision?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

b) For cruise safety planning, would you recommend a lower threshold (0.3) or higher threshold (0.7)? Why?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

___________________________________________________________________

### Question 6.3: Model Limitations

a) What factors that influenced Titanic survival are NOT captured in this dataset?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

b) How would you improve this model if you had more data?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

---

## Part 7: Bonus Visualization (Optional)

**TODO:** Create a probability distribution plot.

In [ ]:
# BONUS: Plot distribution of predicted probabilities
# Hint: Separate probabilities by actual class, plot overlapping histograms

prob_survived = y_pred_proba[:, 1]

# Separate by actual class
prob_died_actual = prob_survived[y_test == 0]
prob_survived_actual = prob_survived[y_test == 1]

# Plot
plt.figure(figsize=(10, 6))
plt.hist(prob_died_actual, bins=20, alpha=0.6, 
         label='Actual: Died', color='red')
plt.hist(prob_survived_actual, bins=20, alpha=0.6, 
         label='Actual: Survived', color='green')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, 
            label='Default Threshold (0.5)')
plt.xlabel('Predicted Probability of Survival')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Probabilities by Actual Outcome')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🎉 Congratulations!

You've completed the Week 2 post-class exercise!

**What you accomplished:**
1. ✅ Loaded and prepared a real-world dataset with categorical features
2. ✅ Handled missing values appropriately
3. ✅ Encoded categorical variables (Sex, Embarked)
4. ✅ Built a complete classification pipeline from scratch
5. ✅ Calculated all major classification metrics
6. ✅ Generated and interpreted confusion matrix
7. ✅ Created professional visualizations (ROC curve, heatmap)
8. ✅ Analyzed model performance across different passenger groups
9. ✅ Thought critically about business context and model limitations

---

## Next Steps

**To deepen your learning:**
1. **Compare with solutions:** Check `week2_postclass_solutions.ipynb`
2. **Try the bonus exercise:** `week2_bonus_threshold_tuning.ipynb` (if available)
3. **Complete self-assessment:** `Week2_Self_Assessment.md`
4. **Review challenging concepts:** Revisit Day 2 materials

**For Week 3 prep:**
- Ensure you can draw confusion matrix from memory
- Practice explaining precision vs recall to a friend
- Review threshold tuning concepts
- Get ready for Decision Trees!

---

## Reflection

**Before you finish, take a moment to reflect:**

1. What was the most challenging part of this exercise?

2. What concept do you feel most confident about?

3. How does handling categorical features differ from numeric features?

4. How comfortable are you with the complete classification pipeline (1-10)?

---

**Great work! See you in Week 3!** 🎓

---

*Week 2 Post-Class Exercise v1.0 | December 2025*